In [ ]:

from datasets import load_dataset



In [ ]:
import os
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from PIL import Image
from transformers import (
    CvtForImageClassification,
    AutoImageProcessor,
    Trainer,
    TrainingArguments
)
from sklearn.model_selection import train_test_split


In [ ]:
import numpy as np


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [ ]:
dataset = load_dataset("Santhosh2312/real-bogus-test_30k")

print(dataset)
print(dataset["train"][0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 29067
    })
})
{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=950x362 at 0x7814ACFFE3F0>, 'label': 0}


In [ ]:
model_name = "microsoft/cvt-13"
processor = AutoImageProcessor.from_pretrained(model_name)


The image processor of type `ConvNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [ ]:
def transform(example):

    images = example["image"]
    labels = example["label"]

    processed_images = []

    for img in images:

        # Split stitched image FIRST (while still PIL)
        width, height = img.size
        split_width = width // 3

        panel1 = img.crop((0, 0, split_width, height)).convert("L")
        panel2 = img.crop((split_width, 0, 2*split_width, height)).convert("L")
        panel3 = img.crop((2*split_width, 0, width, height)).convert("L")

        # Now process each panel separately
        panel1 = processor(images=panel1, return_tensors="pt")["pixel_values"].squeeze(0)
        panel2 = processor(images=panel2, return_tensors="pt")["pixel_values"].squeeze(0)
        panel3 = processor(images=panel3, return_tensors="pt")["pixel_values"].squeeze(0)

        # Each panel is now (3, 224, 224) but channels are identical
        # Keep only ONE channel from each
        panel1 = panel1[0:1, :, :]
        panel2 = panel2[0:1, :, :]
        panel3 = panel3[0:1, :, :]

        # Stack as 3 channels
        combined = torch.cat([panel1, panel2, panel3], dim=0)

        processed_images.append(combined)

    return {
        "pixel_values": torch.stack(processed_images),
        "labels": torch.tensor(labels)
    }


In [ ]:
full_dataset = dataset["train"]

labels = full_dataset["label"]

train_indices, test_indices = train_test_split(
    np.arange(len(full_dataset)),
    test_size=200,
    stratify=labels,
    random_state=42
)

train_dataset = full_dataset.select(train_indices)
test_dataset = full_dataset.select(test_indices)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))


Train size: 28867
Test size: 200


In [ ]:
labels_train = train_dataset["label"]

train_indices, val_indices = train_test_split(
    np.arange(len(train_dataset)),
    test_size=0.01,
    stratify=labels_train,
    random_state=42
)

final_train_dataset = train_dataset.select(train_indices)
val_dataset = train_dataset.select(val_indices)

print("Final train size:", len(final_train_dataset))
print("Validation size:", len(val_dataset))


Final train size: 28578
Validation size: 289


In [ ]:
final_train_dataset = final_train_dataset.with_transform(transform)
val_dataset = val_dataset.with_transform(transform)
test_dataset = test_dataset.with_transform(transform)

In [ ]:
model = CvtForImageClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True,
    id2label={0: "bogus", 1: "real"},
    label2id={"bogus": 0, "real": 1}
)


Loading weights:   0%|          | 0/459 [00:00<?, ?it/s]

CvtForImageClassification LOAD REPORT from: microsoft/cvt-13
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 384]) vs model:torch.Size([2, 384])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    remove_unused_columns=False   # 🔥 THIS LINE FIXES IT
)


In [ ]:
def custom_data_collator(features):
    batch = {}
    batch["pixel_values"] = torch.stack([f["pixel_values"] for f in features])
    batch["labels"] = torch.tensor([f["labels"] for f in features])
    return batch

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_train_dataset,
    eval_dataset=val_dataset,
    data_collator=custom_data_collator
)

In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.225319,0.051966
2,0.156525,0.004191
3,0.123200,0.022028
4,0.089151,0.008032
5,0.074905,0.000281


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=17865, training_loss=0.14875619302702123, metrics={'train_runtime': 4086.3592, 'train_samples_per_second': 34.968, 'train_steps_per_second': 4.372, 'total_flos': 2.53116200139264e+18, 'train_loss': 0.14875619302702123, 'epoch': 5.0})

In [ ]:
print(dataset["train"].column_names)


['image', 'label']


In [ ]:
trainer.evaluate(test_dataset)


{'eval_loss': 0.030110539868474007,
 'eval_runtime': 3.456,
 'eval_samples_per_second': 57.87,
 'eval_steps_per_second': 7.234,
 'epoch': 5.0}

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Get predictions
predictions = trainer.predict(test_dataset)

logits = predictions.predictions
labels = predictions.label_ids

# Convert logits to predicted class
preds = np.argmax(logits, axis=-1)

# Compute metrics
accuracy = accuracy_score(labels, preds)
precision = precision_score(labels, preds)
recall = recall_score(labels, preds)
f1 = f1_score(labels, preds)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")


Accuracy : 0.9900
Precision: 0.9907
Recall   : 0.9907
F1 Score : 0.9907


In [ ]:
trainer.save_model("cvt_real_bogus_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
processor.save_pretrained("cvt_real_bogus_model")


['cvt_real_bogus_model/preprocessor_config.json']

In [ ]:
!zip -r cvt_real_bogus_model.zip cvt_real_bogus_model


  adding: cvt_real_bogus_model/ (stored 0%)
  adding: cvt_real_bogus_model/preprocessor_config.json (deflated 43%)
  adding: cvt_real_bogus_model/training_args.bin (deflated 53%)
  adding: cvt_real_bogus_model/model.safetensors (deflated 7%)
  adding: cvt_real_bogus_model/config.json (deflated 64%)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')